In [0]:
# CELL 1: Install & Verify Libraries
# -------------------------------------------------------------------------
# 1. Install the library
%pip install beautifulsoup4
 
# 2. Verify installation immediately
try:
    from bs4 import BeautifulSoup
    print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:
    print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")

In [0]:
# CELL 2: Setup & Dynamic Link Finder
# -------------------------------------------------------------------------
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, timedelta
 
# 1. Define URL
base_page_url = "https://www.dubaipulse.gov.ae/data/dld-transactions/dld_transactions-open"
 
# -------------------------------------------------------------------------
# 2. Azure Storage Paths (PASTE YOUR INFO BELOW)
# -------------------------------------------------------------------------
storage_account_name = "REMOVED_FOR_SECURITY"
storage_account_key = "REMOVED_FOR_SECURITY"
 
# Connect to ADLS
try:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
 
    # ✨ AUTO-CREATE FIX: This creates the 'bronze' container if it's missing
    spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")
    print("Configuration Loaded & Auto-Create Enabled.")
except Exception as e:
    if "CONFIG_NOT_AVAILABLE" not in str(e):
        raise
 
# Define Folders
landing_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/"
archive_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/archive/"
 
# 3. Dynamic Link Finder
def get_dynamic_download_url():
    print(f"🔍 Scraping {base_page_url}...")
    headers = {'User-Agent': 'Mozilla/5.0'}
 
    response = requests.get(base_page_url, headers=headers)
    response.raise_for_status()
 
    soup = BeautifulSoup(response.content, 'html.parser')
 
# Logic: Find link containing 'download' and 'transactions.csv'
    for link in soup.find_all('a', href=True):
        href = link['href']
        if "download" in href and "transactions.csv" in href:
             final_url = f"https://www.dubaipulse.gov.ae{href}" if href.startswith("/") else href
             print(f"✅ Found dynamic link: {final_url}")
             return final_url
 
    raise Exception("❌ Could not find the download link on the page.")

In [0]:
# CELL 4: Archival Logic
def archive_current_file():
    current_file_path = f"{landing_zone}current.csv"
    try:
        dbutils.fs.ls(current_file_path)
        today_str    = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone}{today_str}.csv"
        print(f"📦 Archiving previous 'current.csv' to: {archive_path}")
        dbutils.fs.mv(current_file_path, archive_path)
    except Exception:
        print("ℹ️ No 'current.csv' found in landing. First run?")

archive_current_file()

In [0]:
files = dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/")
for f in files:
    print(f.name, f.size)